# Experiment 07 — Live Rapid-E Model Evaluation Analysis

This notebook analyzes the outputs from `exp07_live_experiment_evaluation.py`.

It compares:
- Experiment 5 closed-set multimodal model
- Experiment 6 robust model with unknown rejection

Main questions:
1. Do controls get rejected as unknown?
2. Do mixture experiments recover the expected Bacillus / Micrococcus proportions?
3. Is model failure linked to low confidence, high entropy, weak fluorescence, or distribution shift?

In [6]:
import pandas as pd
from pathlib import Path

summary_path = Path(
    r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\all_experiment_summary.csv"
)

summary = pd.read_csv(summary_path)
summary.head()

,model_name,experiment_date,experiment_id,experiment_block,n_particles,unknown_fraction,median_confidence,median_entropy,median_peak_fluorescence,fraction_above_2000,median_size,median_time_asymmetry,frac_B_endophyticus,frac_K_salsicia,frac_S_huminis,frac_B_cereus,frac_M_luteus
0,exp05_multimodal,2026-06-17,bacillus_cereus_possible_1224_1246,bacillus_cereus_possible_1224_1246,104436,0.0,0.999993,8.891467e-05,436.0,0.000268,2.444172,0.086134,0.987849,0.009614,0.002356,0.000134,0.000048
1,exp06_robust,2026-06-17,bacillus_cereus_possible_1224_1246,bacillus_cereus_possible_1224_1246,104436,0.0,1.000000,3.001001e-07,436.0,0.000268,2.444172,0.086134,0.995595,0.001264,0.002873,0.000077,0.000192
2,exp05_multimodal,2026-06-17,distilled_water_1107_1223,distilled_water_1107_1223,67383,0.0,1.000000,5.305745e-06,455.0,0.000326,0.766009,0.112199,0.992179,0.005743,0.001914,0.000074,0.000089
3,exp06_robust,2026-06-17,distilled_water_1107_1223,distilled_water_1107_1223,67383,0.0,1.000000,1.372806e-10,455.0,0.000326,0.766009,0.112199,0.994613,0.001440,0.003621,0.000015,0.000312
4,exp05_multimodal,2026-06-17,distilled_water_1248_1320,distilled_water_1248_1320,35206,0.0,1.000000,1.510151e-06,435.0,0.000369,0.120900,0.136954,0.995114,0.003607,0.001250,0.000028,NaN


In [10]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(
    Path(r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp06_robust\2026-06-17\distilled_water_1107_1223\particle_predictions.parquet")
)

df.columns

Index(['model_name', 'experiment_date', 'experiment_id', 'experiment_block', 'raw_file', 'source_file', 'particle_index', 'predicted_label',
       'prediction_confidence', 'entropy', 'robust_prediction', 'unknown_flag', 'peak_fluorescence', 'size', 'time_asymmetry', 'prob_B_cereus',
       'prob_B_endophyticus', 'prob_K_salsicia', 'prob_M_luteus', 'prob_S_huminis'],
      dtype='str')

In [8]:
from pathlib import Path

root = Path(
    r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp06_robust"
)

for p in sorted(root.rglob("particle_predictions.parquet")):
    print(p)

C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp06_robust\2026-06-17\bacillus_cereus_possible_1224_1246\particle_predictions.parquet
C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp06_robust\2026-06-17\distilled_water_1107_1223\particle_predictions.parquet
C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp06_robust\2026-06-17\distilled_water_1248_1320\particle_predictions.parquet
C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp06_robust\2026-06-17\distilled_water_1346_1420\particle_predictions.parquet
C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp06_robust\2026-06-17\micrococcus_1323_1345\particle_predictions.parquet
C:\Users\chris\OneDrive\Docume

In [9]:
from pathlib import Path

root = Path(r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation")

for p in sorted(root.rglob("experiment_summary.csv")):
    print(p)

C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp05_multimodal\2026-06-17\bacillus_cereus_possible_1224_1246\experiment_summary.csv
C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp05_multimodal\2026-06-17\distilled_water_1107_1223\experiment_summary.csv
C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp05_multimodal\2026-06-17\distilled_water_1248_1320\experiment_summary.csv
C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp05_multimodal\2026-06-17\distilled_water_1346_1420\experiment_summary.csv
C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp07_model_evaluation\exp05_multimodal\2026-06-17\micrococcus_1323_1345\experiment_summary.csv
C:\Users\chris\OneDrive\Documents\Univer

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_ROOT = Path("results/live_rapid_e/exp07_model_evaluation")
SUMMARY_PATH = RESULTS_ROOT / "all_experiment_summary.csv"
PARTICLE_PARQUET = RESULTS_ROOT / "all_particle_predictions.parquet"
PARTICLE_CSV = RESULTS_ROOT / "all_particle_predictions.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Results root:", RESULTS_ROOT.resolve())
print("Summary exists:", SUMMARY_PATH.exists())
print("Particle parquet exists:", PARTICLE_PARQUET.exists())
print("Particle CSV exists:", PARTICLE_CSV.exists())

Results root: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\reports\notebooks\results\live_rapid_e\exp07_model_evaluation
Summary exists: False
Particle parquet exists: False
Particle CSV exists: False


## 1. Load Experiment 07 Results

In [4]:
summary = pd.read_csv(SUMMARY_PATH)

if PARTICLE_PARQUET.exists():
    particles = pd.read_parquet(PARTICLE_PARQUET)
else:
    particles = pd.read_csv(PARTICLE_CSV)

print("Summary shape:", summary.shape)
print("Particle shape:", particles.shape)

display(summary.head())
display(particles.head())

FileNotFoundError: [Errno 2] No such file or directory: 'results\\live_rapid_e\\exp07_model_evaluation\\all_experiment_summary.csv'

## 2. Basic Data Checks

In [ ]:
required_particle_cols = [
    "model_name",
    "experiment_date",
    "experiment_id",
    "experiment_block",
    "predicted_label",
    "prediction_confidence",
    "entropy",
    "robust_prediction",
    "unknown_flag",
    "peak_fluorescence",
    "size",
    "time_asymmetry",
]

missing = [c for c in required_particle_cols if c not in particles.columns]

if missing:
    print("Missing columns:", missing)
else:
    print("All required particle columns present.")

print("\nModels:")
display(particles["model_name"].value_counts())

print("\nExperiments:")
display(
    particles[["experiment_date", "experiment_id", "experiment_block"]]
    .drop_duplicates()
    .sort_values(["experiment_date", "experiment_id"])
)

## 3. Add Ground Truth Metadata for Controls and Mixtures

In [ ]:
def expected_mix_from_block(block: str):
    block = str(block).lower()

    if "bacillus75_micrococcus25" in block:
        return 75, 25, "mixture"
    if "bacillus50_micrococcus50" in block:
        return 50, 50, "mixture"
    if "bacillus25_micrococcus75" in block:
        return 25, 75, "mixture"

    if "bacillus_cereus" in block:
        return 100, 0, "single_species"
    if "micrococcus" in block:
        return 0, 100, "single_species"

    if "distilled" in block:
        return 0, 0, "control"
    if "ringer" in block:
        return 0, 0, "control"

    return np.nan, np.nan, "unknown"

meta = (
    particles[["experiment_date", "experiment_id", "experiment_block"]]
    .drop_duplicates()
    .copy()
)

meta[["expected_bacillus_pct", "expected_micrococcus_pct", "experiment_type"]] = (
    meta["experiment_block"]
    .apply(lambda x: pd.Series(expected_mix_from_block(x)))
)

particles = particles.merge(
    meta,
    on=["experiment_date", "experiment_id", "experiment_block"],
    how="left",
)

summary = summary.merge(
    meta,
    on=["experiment_date", "experiment_id", "experiment_block"],
    how="left",
)

display(meta.sort_values(["experiment_date", "experiment_id"]))

## 4. Particle Counts and Fluorescence Threshold Behavior

In [ ]:
quality = (
    particles
    .groupby(["model_name", "experiment_date", "experiment_id", "experiment_block", "experiment_type"], dropna=False)
    .agg(
        n_particles=("predicted_label", "size"),
        median_peak_fluorescence=("peak_fluorescence", "median"),
        q95_peak_fluorescence=("peak_fluorescence", lambda x: np.nanpercentile(x, 95)),
        q99_peak_fluorescence=("peak_fluorescence", lambda x: np.nanpercentile(x, 99)),
        fraction_above_2000=("peak_fluorescence", lambda x: np.nanmean(x > 2000)),
        median_size=("size", "median"),
        median_time_asymmetry=("time_asymmetry", "median"),
    )
    .reset_index()
)

display(
    quality
    .sort_values(["experiment_date", "experiment_id", "model_name"])
)

### Plot: Fraction of Particles Above 2000 a.u.

In [ ]:
plot_df = (
    quality
    .drop_duplicates(["model_name", "experiment_id"])
    .sort_values(["experiment_date", "experiment_id", "model_name"])
)

for model_name, df in plot_df.groupby("model_name"):
    plt.figure(figsize=(12, 5))
    plt.bar(df["experiment_id"] + "\n" + df["experiment_block"].str.slice(0, 25), df["fraction_above_2000"])
    plt.xticks(rotation=90)
    plt.ylabel("Fraction above 2000 a.u.")
    plt.title(f"Fluorescence threshold behavior — {model_name}")
    plt.tight_layout()
    plt.show()

## 5. Prediction Composition by Experiment

In [ ]:
composition = (
    particles
    .groupby(["model_name", "experiment_date", "experiment_id", "experiment_block", "robust_prediction"])
    .size()
    .reset_index(name="n")
)

composition["fraction"] = (
    composition["n"] /
    composition.groupby(["model_name", "experiment_date", "experiment_id", "experiment_block"])["n"].transform("sum")
)

composition_pivot = composition.pivot_table(
    index=["model_name", "experiment_date", "experiment_id", "experiment_block"],
    columns="robust_prediction",
    values="fraction",
    fill_value=0,
).reset_index()

display(composition_pivot.sort_values(["experiment_date", "experiment_id", "model_name"]))

### Plot: Prediction Composition

In [ ]:
for model_name, df in composition.groupby("model_name"):
    pivot = df.pivot_table(
        index=["experiment_id", "experiment_block"],
        columns="robust_prediction",
        values="fraction",
        fill_value=0,
    )

    pivot = pivot.sort_index()

    plt.figure(figsize=(12, 6))
    bottom = np.zeros(len(pivot))

    labels = list(pivot.columns)
    x = np.arange(len(pivot))

    for label in labels:
        values = pivot[label].to_numpy()
        plt.bar(x, values, bottom=bottom, label=str(label))
        bottom += values

    tick_labels = [f"{idx[0]}\n{idx[1][:22]}" for idx in pivot.index]
    plt.xticks(x, tick_labels, rotation=90)
    plt.ylabel("Prediction fraction")
    plt.title(f"Prediction composition by experiment — {model_name}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## 6. Unknown Rejection Analysis

In [ ]:
unknown_summary = (
    particles
    .groupby(["model_name", "experiment_date", "experiment_id", "experiment_block", "experiment_type"], dropna=False)
    .agg(
        n_particles=("unknown_flag", "size"),
        unknown_fraction=("unknown_flag", "mean"),
        median_confidence=("prediction_confidence", "median"),
        median_entropy=("entropy", "median"),
        fraction_above_2000=("peak_fluorescence", lambda x: np.nanmean(x > 2000)),
    )
    .reset_index()
)

display(
    unknown_summary
    .sort_values(["experiment_date", "experiment_id", "model_name"])
)

### Plot: Unknown Fraction

In [ ]:
for model_name, df in unknown_summary.groupby("model_name"):
    plt.figure(figsize=(12, 5))
    plt.bar(df["experiment_id"] + "\n" + df["experiment_block"].str.slice(0, 25), df["unknown_fraction"])
    plt.xticks(rotation=90)
    plt.ylabel("Unknown fraction")
    plt.ylim(0, 1)
    plt.title(f"Unknown rejection by experiment — {model_name}")
    plt.tight_layout()
    plt.show()

## 7. Confidence and Entropy Distributions

In [ ]:
for model_name, df_model in particles.groupby("model_name"):
    plt.figure(figsize=(10, 5))
    for exp_type, df_type in df_model.groupby("experiment_type"):
        values = df_type["prediction_confidence"].dropna()
        if len(values):
            plt.hist(values, bins=40, alpha=0.5, label=exp_type)
    plt.xlabel("Prediction confidence")
    plt.ylabel("Particle count")
    plt.title(f"Confidence distribution by experiment type — {model_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    for exp_type, df_type in df_model.groupby("experiment_type"):
        values = df_type["entropy"].dropna()
        if len(values):
            plt.hist(values, bins=40, alpha=0.5, label=exp_type)
    plt.xlabel("Entropy")
    plt.ylabel("Particle count")
    plt.title(f"Entropy distribution by experiment type — {model_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 8. Mixture Recovery Analysis

In [ ]:
def is_bacillus_label(label):
    label = str(label).lower()
    return "cereus" in label or "bacillus" in label

def is_micrococcus_label(label):
    label = str(label).lower()
    return "luteus" in label or "micrococcus" in label

mix_particles = particles[particles["experiment_type"].isin(["mixture", "single_species"])].copy()

mix_particles["is_predicted_bacillus"] = mix_particles["robust_prediction"].apply(is_bacillus_label)
mix_particles["is_predicted_micrococcus"] = mix_particles["robust_prediction"].apply(is_micrococcus_label)
mix_particles["is_known"] = ~mix_particles["unknown_flag"].astype(bool)

mixture_recovery = (
    mix_particles
    .groupby([
        "model_name",
        "experiment_date",
        "experiment_id",
        "experiment_block",
        "expected_bacillus_pct",
        "expected_micrococcus_pct",
    ], dropna=False)
    .agg(
        n_particles=("robust_prediction", "size"),
        unknown_fraction=("unknown_flag", "mean"),
        predicted_bacillus_all_particles=("is_predicted_bacillus", "mean"),
        predicted_micrococcus_all_particles=("is_predicted_micrococcus", "mean"),
    )
    .reset_index()
)

# Conditional composition among known / non-unknown particles
known_mix = mix_particles[mix_particles["is_known"]].copy()

known_recovery = (
    known_mix
    .groupby([
        "model_name",
        "experiment_date",
        "experiment_id",
        "experiment_block",
    ], dropna=False)
    .agg(
        n_known_particles=("robust_prediction", "size"),
        predicted_bacillus_known_particles=("is_predicted_bacillus", "mean"),
        predicted_micrococcus_known_particles=("is_predicted_micrococcus", "mean"),
    )
    .reset_index()
)

mixture_recovery = mixture_recovery.merge(
    known_recovery,
    on=["model_name", "experiment_date", "experiment_id", "experiment_block"],
    how="left",
)

for col in [
    "predicted_bacillus_all_particles",
    "predicted_micrococcus_all_particles",
    "predicted_bacillus_known_particles",
    "predicted_micrococcus_known_particles",
]:
    mixture_recovery[col] = mixture_recovery[col] * 100

display(mixture_recovery.sort_values(["model_name", "expected_bacillus_pct"], ascending=[True, False]))

### Plot: True vs Predicted Bacillus Proportion

In [ ]:
mix_only = mixture_recovery[
    mixture_recovery["experiment_block"].str.contains("bacillus", case=False, na=False)
].copy()

for model_name, df in mix_only.groupby("model_name"):
    df = df.sort_values("expected_bacillus_pct")

    plt.figure(figsize=(6, 6))
    plt.plot([0, 100], [0, 100], linestyle="--", label="Ideal")
    plt.scatter(df["expected_bacillus_pct"], df["predicted_bacillus_all_particles"], label="All particles")
    if "predicted_bacillus_known_particles" in df:
        plt.scatter(df["expected_bacillus_pct"], df["predicted_bacillus_known_particles"], label="Known only")
    plt.xlabel("Expected Bacillus (%)")
    plt.ylabel("Predicted Bacillus (%)")
    plt.title(f"Mixture recovery — {model_name}")
    plt.xlim(0, 100)
    plt.ylim(0, 100)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9. Controls: Do Water and Ringer Get Rejected?

In [ ]:
controls = particles[particles["experiment_type"] == "control"].copy()

control_summary = (
    controls
    .groupby(["model_name", "experiment_date", "experiment_id", "experiment_block"])
    .agg(
        n_particles=("robust_prediction", "size"),
        unknown_fraction=("unknown_flag", "mean"),
        median_confidence=("prediction_confidence", "median"),
        median_entropy=("entropy", "median"),
        fraction_above_2000=("peak_fluorescence", lambda x: np.nanmean(x > 2000)),
    )
    .reset_index()
)

display(control_summary.sort_values(["experiment_date", "experiment_id", "model_name"]))

control_comp = (
    controls
    .groupby(["model_name", "experiment_id", "experiment_block", "robust_prediction"])
    .size()
    .reset_index(name="n")
)

control_comp["fraction"] = (
    control_comp["n"] /
    control_comp.groupby(["model_name", "experiment_id", "experiment_block"])["n"].transform("sum")
)

display(control_comp.sort_values(["model_name", "experiment_id", "fraction"], ascending=[True, True, False]))

## 10. Identify Potential Distribution Shift

In [ ]:
shift_summary = (
    particles
    .groupby(["model_name", "experiment_type"])
    .agg(
        n_particles=("robust_prediction", "size"),
        median_peak_fluorescence=("peak_fluorescence", "median"),
        q95_peak_fluorescence=("peak_fluorescence", lambda x: np.nanpercentile(x, 95)),
        fraction_above_2000=("peak_fluorescence", lambda x: np.nanmean(x > 2000)),
        median_size=("size", "median"),
        median_time_asymmetry=("time_asymmetry", "median"),
        median_confidence=("prediction_confidence", "median"),
        median_entropy=("entropy", "median"),
        unknown_fraction=("unknown_flag", "mean"),
    )
    .reset_index()
)

display(shift_summary)

## 11. Export Thesis-Ready Tables

In [ ]:
ANALYSIS_OUT = RESULTS_ROOT / "analysis_tables"
ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)

quality.to_csv(ANALYSIS_OUT / "data_quality_by_experiment.csv", index=False)
composition_pivot.to_csv(ANALYSIS_OUT / "prediction_composition_by_experiment.csv", index=False)
unknown_summary.to_csv(ANALYSIS_OUT / "unknown_rejection_summary.csv", index=False)
mixture_recovery.to_csv(ANALYSIS_OUT / "mixture_recovery_summary.csv", index=False)
control_summary.to_csv(ANALYSIS_OUT / "control_rejection_summary.csv", index=False)
shift_summary.to_csv(ANALYSIS_OUT / "distribution_shift_summary.csv", index=False)

print("Saved analysis tables to:", ANALYSIS_OUT)

## 12. Interpretation Checklist

Use the outputs above to answer these thesis questions:

1. **Controls:** Are distilled water and Ringer mostly rejected as unknown by Exp06?
2. **Closed-set failure:** Does Exp05 force controls into bacterial labels?
3. **Mixture recovery:** Do the 75/25, 50/50, and 25/75 experiments produce monotonic Bacillus/Micrococcus trends?
4. **Unknown-aware recovery:** For Exp06, does mixture recovery improve when considering only known particles?
5. **Distribution shift:** Are low confidence, high entropy, or abnormal fluorescence distributions associated with failure?
6. **Experimental quality:** Is the fraction above 2000 a.u. comparable across controls and bacterial aerosols?